# 11 - Dashboard Data Preparation

## Objective

Prepare dashboard-ready datasets for Streamlit and the optional FastAPI layer. The consolidated records preserve the final research mapping: statistical evidence for RQ1–RQ3, SHAP outputs for RQ4, and operational prioritization results for RQ5.


#### Load project configuration

In [1]:
from __future__ import annotations

import importlib.util
import io
import json
import os
from pathlib import Path

from databricks.sdk import WorkspaceClient
from dotenv import dotenv_values

bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not bootstrap.is_file():
    bootstrap = Path.cwd() / "import_path.py"
spec = importlib.util.spec_from_file_location("import_path", bootstrap)
import_path = importlib.util.module_from_spec(spec)
spec.loader.exec_module(import_path)

from config import project_config as cfg

DATABRICKS_PROFILE = os.getenv(
    "DATABRICKS_CONFIG_PROFILE",
    "capstone-serverless",
)
api_env_candidates = [
    Path.cwd() / "api" / ".env",
    Path.cwd().parent / "api" / ".env",
]
api_env_path = next(
    (candidate for candidate in api_env_candidates if candidate.is_file()),
    None,
)
api_credentials = dotenv_values(api_env_path) if api_env_path else {}
LOCAL_DATABRICKS_HOST = str(
    api_credentials.get("DATABRICKS_SERVER_HOSTNAME", "")
).strip()
LOCAL_DATABRICKS_TOKEN = str(
    api_credentials.get("DATABRICKS_ACCESS_TOKEN", "")
).strip()
if LOCAL_DATABRICKS_HOST and not LOCAL_DATABRICKS_HOST.startswith("http"):
    LOCAL_DATABRICKS_HOST = f"https://{LOCAL_DATABRICKS_HOST}"

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession

    builder = DatabricksSession.builder.serverless()
    if LOCAL_DATABRICKS_HOST and LOCAL_DATABRICKS_TOKEN:
        builder = builder.host(LOCAL_DATABRICKS_HOST).token(
            LOCAL_DATABRICKS_TOKEN
        )
    else:
        builder = builder.profile(DATABRICKS_PROFILE)
    spark = builder.getOrCreate()


def artifact_client() -> WorkspaceClient:
    try:
        dbutils
    except NameError:
        if LOCAL_DATABRICKS_HOST and LOCAL_DATABRICKS_TOKEN:
            return WorkspaceClient(
                host=LOCAL_DATABRICKS_HOST,
                token=LOCAL_DATABRICKS_TOKEN,
                auth_type="pat",
            )
        return WorkspaceClient(profile=DATABRICKS_PROFILE)
    return WorkspaceClient()


client = artifact_client()

print("Project configuration and Spark session loaded successfully.")
print(f"Dashboard table: {cfg.DASHBOARD_TABLE}")
print(f"Explorer table: {cfg.DASHBOARD_EXPLORER_TABLE}")
print(f"Insights table: {cfg.DASHBOARD_INSIGHTS_TABLE}")


Project configuration and Spark session loaded successfully.
Dashboard table: workspace.default.flight_dashboard
Explorer table: workspace.default.flight_dashboard_explorer
Insights table: workspace.default.flight_dashboard_insights


#### Load source datasets

The dashboard assembly combines cleaned operational history, model predictions, SHAP artifacts, statistical validation outputs, and prioritization evaluation results.

In [2]:
from pyspark.sql import functions as F


def require_table(table_name: str) -> None:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the upstream notebooks before continuing."
        )


def download_artifact(file_path: str) -> bytes:
    try:
        response = client.files.download(file_path)
        if response.contents is None:
            raise RuntimeError("The downloaded artifact is empty.")
        return response.contents.read()
    except Exception as exc:
        raise RuntimeError(
            f"Required file '{file_path}' was not found. "
            "Run the upstream notebooks before continuing."
        ) from exc


required_tables = [
    cfg.CLEAN_TABLE,
    cfg.PREDICTIONS_TABLE,
    cfg.STATISTICAL_RESULTS_TABLE,
    cfg.SHAP_GLOBAL_IMPORTANCE_TABLE,
    cfg.SHAP_DIRECTION_EFFECTS_TABLE,
    cfg.PRIORITIZATION_EVALUATION_TABLE,
]
for table_name in required_tables:
    require_table(table_name)

model_metrics = json.loads(
    download_artifact(cfg.SELECTED_MODEL_METRICS_PATH).decode("utf-8")
)
df_clean = spark.table(cfg.CLEAN_TABLE)
df_predictions = spark.table(cfg.PREDICTIONS_TABLE)
df_statistical = spark.table(cfg.STATISTICAL_RESULTS_TABLE)
df_shap_global = spark.table(cfg.SHAP_GLOBAL_IMPORTANCE_TABLE)
df_shap_direction = spark.table(cfg.SHAP_DIRECTION_EFFECTS_TABLE)
df_prioritization_eval = spark.table(
    cfg.PRIORITIZATION_EVALUATION_TABLE
)

print("Dashboard source datasets loaded successfully.")
print(f"Predictions rows: {df_predictions.count():,}")
print(f"SHAP features: {df_shap_global.count():,}")


Dashboard source datasets loaded successfully.
Predictions rows: 500,000
SHAP features: 19


#### Build overview and historical dashboard metrics

In [3]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

import pandas as pd

from utils.dashboard_preparation import (
    MONTH_LABELS,
    build_delay_cause_records,
    build_model_metric_records,
    build_monthly_trend_records,
    build_overview_kpi_records,
    build_research_validation_records,
    combine_dashboard_records,
    serialize_dashboard_metadata,
)


eligible_flights = df_clean.filter(
    (F.col(cfg.CANCELLED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
    & (F.col(cfg.DIVERTED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
)


eligible_overview_row = eligible_flights.agg(
    F.count("*").alias("total_flights"),
    (
        F.avg(cfg.TARGET_COLUMN) * 100
    ).alias("delay_rate"),
    F.avg(cfg.ARRIVAL_DELAY_COLUMN).alias(
        "average_arrival_delay"
    ),
).collect()[0]


cancellation_row = df_clean.agg(
    (
        F.avg(
            F.col(cfg.CANCELLED_COLUMN).cast("double")
        ) * 100
    ).alias("cancellation_rate")
).collect()[0]

overview_records = build_overview_kpi_records(
    total_flights=int(
        eligible_overview_row["total_flights"]
    ),
    delay_rate=float(
        eligible_overview_row["delay_rate"] or 0.0
    ),
    average_arrival_delay=float(
        eligible_overview_row["average_arrival_delay"] or 0.0
    ),
    cancellation_rate=float(
        cancellation_row["cancellation_rate"] or 0.0
    ),
)

monthly_trend_pdf = (
    eligible_flights.groupBy(cfg.MONTH_COLUMN)
    .agg((F.avg(cfg.TARGET_COLUMN) * 100).alias("delay_rate"))
    .orderBy(cfg.MONTH_COLUMN)
    .toPandas()
)
monthly_trend_pdf["month_label"] = monthly_trend_pdf[cfg.MONTH_COLUMN].map(MONTH_LABELS)
monthly_trend_pdf["month_number"] = monthly_trend_pdf[cfg.MONTH_COLUMN]
monthly_records = build_monthly_trend_records(monthly_trend_pdf)

cause_expressions = [
    F.sum(F.col(column_name)).alias(column_name)
    for column_name in cfg.DELAY_CAUSE_COLUMNS
]
cause_totals = {
    key: float(value or 0.0)
    for key, value in (
        eligible_flights.agg(*cause_expressions).collect()[0].asDict()
    ).items()
}
total_delay_minutes = float(sum(cause_totals.values())) or 1.0
cause_rows = []
for column_name, label in cfg.DELAY_CAUSE_LABELS.items():
    cause_rows.append(
        {
            "cause": label,
            "percentage": round((cause_totals[column_name] / total_delay_minutes) * 100, 2),
        }
    )
cause_records = build_delay_cause_records(pd.DataFrame(cause_rows))

display(spark.createDataFrame(overview_records))
display(spark.createDataFrame(monthly_records))


,section,metric_name,metric_value,metric_text,dimension_1,dimension_2,sort_order
0,overview_kpi,total_flights,6.879483e+06,6879483,None,None,1
1,overview_kpi,avg_delay_rate,2.230745e+01,22.3%,None,None,2
2,overview_kpi,avg_arr_delay,8.504474e+00,8.5 min,None,None,3
3,overview_kpi,cancel_rate,1.469276e+00,1.47%,None,None,4


,section,metric_name,metric_value,metric_text,dimension_1,dimension_2,sort_order
0,monthly_trend,delay_rate,18.789168,Jan,Jan,None,1
1,monthly_trend,delay_rate,20.766764,Feb,Feb,None,2
2,monthly_trend,delay_rate,19.590377,Mar,Mar,None,3
3,monthly_trend,delay_rate,19.663857,Apr,Apr,None,4
4,monthly_trend,delay_rate,23.601763,May,May,None,5
5,monthly_trend,delay_rate,28.259035,Jun,Jun,None,6
6,monthly_trend,delay_rate,28.885252,Jul,Jul,None,7
7,monthly_trend,delay_rate,22.555593,Aug,Aug,None,8
8,monthly_trend,delay_rate,16.631622,Sep,Sep,None,9
9,monthly_trend,delay_rate,20.321991,Oct,Oct,None,10


#### Build explorer, insights, and research-validation datasets

`RiskTier` is copied directly from the validation-derived `risk_level` created
by Notebook 10. This notebook does not recalculate or override risk bands.

In [4]:
predictions_pdf = df_predictions.toPandas()

explorer_pdf = predictions_pdf.assign(
    Flight=predictions_pdf["flight_label"],
    Carrier=predictions_pdf["airline_code"],
    Origin=predictions_pdf["origin_airport"],
    Destination=predictions_pdf["destination_airport"],
    SchedDep=predictions_pdf["scheduled_departure_text"],
    DepartureWindow=predictions_pdf["departure_window"],
    DepTime=predictions_pdf["scheduled_departure_text"],
    DelayProb=predictions_pdf["delay_probability"],
    DelayProbPct=(predictions_pdf["delay_probability"] * 100).round(1).astype(str) + "%",
    RiskTier=predictions_pdf["risk_level"],
    Status=predictions_pdf["risk_level"],
    Month=predictions_pdf["month_number"].map(MONTH_LABELS),
    ShapMainDriver=predictions_pdf["shap_main_driver"],
)

insights_global_pdf = df_shap_global.toPandas().rename(
    columns={"Feature": "feature", "MeanAbsSHAP": "importance"}
)
insights_direction_pdf = df_shap_direction.toPandas()

statistical_pdf = df_statistical.toPandas()
statistical_records = build_research_validation_records(
    statistical_pdf.assign(
        metric_name=statistical_pdf["research_question"],
        metric_value=statistical_pdf["p_value"],
        metric_text=statistical_pdf["decision"],
        dimension_1=statistical_pdf["factor"],
        dimension_2=statistical_pdf["test_name"],
    )
)

prioritization_eval_pdf = df_prioritization_eval.toPandas()
rq5_records = build_research_validation_records(
    prioritization_eval_pdf.assign(
        metric_name="RQ5",
        metric_value=prioritization_eval_pdf["delay_recall"],
        metric_text=prioritization_eval_pdf["strategy"],
        dimension_1=prioritization_eval_pdf["capacity_k"].astype(str),
        dimension_2=prioritization_eval_pdf["strategy"],
    )
)

model_metric_records = build_model_metric_records(model_metrics)
dashboard_records_pdf = combine_dashboard_records(
    [
        overview_records,
        monthly_records,
        cause_records,
        statistical_records,
        rq5_records,
        model_metric_records,
    ]
)

display(spark.createDataFrame(dashboard_records_pdf).limit(20))
display(spark.createDataFrame(explorer_pdf).select(
    "Flight", "Carrier", "Origin", "Destination", "DelayProb", "RiskTier", "Month"
).limit(10))
display(spark.createDataFrame(insights_global_pdf).head(10))


,section,metric_name,metric_value,metric_text,dimension_1,dimension_2,sort_order
0,overview_kpi,total_flights,6.879483e+06,6879483,None,None,1
1,overview_kpi,avg_delay_rate,2.230745e+01,22.3%,None,None,2
2,overview_kpi,avg_arr_delay,8.504474e+00,8.5 min,None,None,3
3,overview_kpi,cancel_rate,1.469276e+00,1.47%,None,None,4
4,monthly_trend,delay_rate,1.878917e+01,Jan,Jan,None,1
5,monthly_trend,delay_rate,2.076676e+01,Feb,Feb,None,2
6,monthly_trend,delay_rate,1.959038e+01,Mar,Mar,None,3
7,monthly_trend,delay_rate,1.966386e+01,Apr,Apr,None,4
8,monthly_trend,delay_rate,2.360176e+01,May,May,None,5
9,monthly_trend,delay_rate,2.825903e+01,Jun,Jun,None,6


,Flight,Carrier,Origin,Destination,DelayProb,RiskTier,Month
0,G4 2877,G4,SFB,GRR,0.082581,LOW,Nov
1,G4 2888,G4,SFB,GRR,0.319773,MEDIUM,Nov
2,G4 289,G4,OAK,BLI,0.189688,LOW,Nov
3,G4 291,G4,BLI,PSP,0.082643,LOW,Nov
4,G4 2915,G4,TOL,SFB,0.219087,MEDIUM,Nov
5,G4 2934,G4,SFB,USA,0.230466,MEDIUM,Nov
6,G4 2937,G4,SFB,TOL,0.111464,LOW,Nov
7,G4 2941,G4,MEM,SFB,0.328866,MEDIUM,Nov
8,G4 2942,G4,HGR,SFB,0.206857,MEDIUM,Nov
9,G4 2949,G4,SYR,SFB,0.246132,MEDIUM,Nov


[Row(feature='Scheduled arrival hour', importance=0.24029435374313907, FeatureColumn='ARR_HOUR'),
 Row(feature='Scheduled departure hour', importance=0.1861671288442365, FeatureColumn='DEP_HOUR'),
 Row(feature='Historical route delay rate', importance=0.1857568436054069, FeatureColumn='ROUTE_HIST_DELAY_RATE'),
 Row(feature='Day of week', importance=0.1244310312341011, FeatureColumn='DAY_OF_WEEK'),
 Row(feature='Month', importance=0.08459019463503063, FeatureColumn='MONTH'),
 Row(feature='Airline', importance=0.08212953249997262, FeatureColumn='OP_UNIQUE_CARRIER'),
 Row(feature='Season', importance=0.07400606077946723, FeatureColumn='SEASON'),
 Row(feature='Destination airport', importance=0.07342739266299877, FeatureColumn='DEST'),
 Row(feature='Origin airport', importance=0.04400687954292389, FeatureColumn='ORIGIN'),
 Row(feature='Historical airline delay rate', importance=0.04365993280564408, FeatureColumn='AIRLINE_HIST_DELAY_RATE')]

#### Save dashboard-ready datasets

In [5]:
dashboard_records_df = spark.createDataFrame(dashboard_records_pdf)
explorer_df = spark.createDataFrame(explorer_pdf)
insights_df = spark.createDataFrame(insights_global_pdf)

(
    dashboard_records_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(cfg.DASHBOARD_DELTA_PATH)
)
(
    dashboard_records_df.writeTo(cfg.DASHBOARD_TABLE).using("delta").createOrReplace()
)

(
    explorer_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(cfg.DASHBOARD_EXPLORER_PATH)
)
(
    explorer_df.writeTo(cfg.DASHBOARD_EXPLORER_TABLE).using("delta").createOrReplace()
)

(
    insights_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(cfg.DASHBOARD_INSIGHTS_PATH)
)
(
    insights_df.writeTo(cfg.DASHBOARD_INSIGHTS_TABLE).using("delta").createOrReplace()
)

dashboard_metadata = {
    "dashboard_table": cfg.DASHBOARD_TABLE,
    "explorer_table": cfg.DASHBOARD_EXPLORER_TABLE,
    "insights_table": cfg.DASHBOARD_INSIGHTS_TABLE,
    "predictions_table": cfg.PREDICTIONS_TABLE,
    "statistical_results_table": cfg.STATISTICAL_RESULTS_TABLE,
    "prioritization_evaluation_table": cfg.PRIORITIZATION_EVALUATION_TABLE,
    "research_questions": cfg.RESEARCH_QUESTIONS,
    "default_capacity_k": cfg.DEFAULT_CAPACITY_K,
}

metadata_content = serialize_dashboard_metadata(
    dashboard_metadata
).encode("utf-8")
client.files.upload(
    cfg.DASHBOARD_METADATA_PATH,
    io.BytesIO(metadata_content),
    overwrite=True,
)

print("Dashboard datasets saved successfully.")
print(f"Dashboard table: {cfg.DASHBOARD_TABLE}")
print(f"Explorer table: {cfg.DASHBOARD_EXPLORER_TABLE}")
print(f"Insights table: {cfg.DASHBOARD_INSIGHTS_TABLE}")
print(f"Metadata file: {cfg.DASHBOARD_METADATA_PATH}")


Dashboard datasets saved successfully.
Dashboard table: workspace.default.flight_dashboard
Explorer table: workspace.default.flight_dashboard_explorer
Insights table: workspace.default.flight_dashboard_insights
Metadata file: /Volumes/workspace/default/flight_delay_capstone/models/dashboard_metadata.json
